# NB SAM IT Solutions — Recruitment Data Cleaning

This notebook documents the data-cleaning process carried out during my
Business Analyst internship at NB SAM IT Solutions Pvt. Ltd.

## Data Confidentiality Notice

The following files contain confidential company and candidate information
and **must not be publicly shared or uploaded to GitHub**:

- `Daily Update.xlsx`
- `Daily_Update_Cleaned.xlsx`
- Any other original or intermediate files containing real candidate-level data

The files are used only for the internship analysis. The original source data
is not included in the public repository.

In [ ]:
import pandas as pd
import re

# Loading Sheet3 from the uploaded excel file
df = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')

# Printing the first 3 rows to verify if it loaded correctly
print("Data loaded successfully! Here is a preview:")
print(df.head(3))

# Dataset dimensions
num_rows = df.shape[0]
num_cols = df.shape[1]
total_entries = df.size
memory_size = df.memory_usage(deep=True).sum() / 1024  # KB

print("\nDataset Information:")
print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_cols}")
print(f"Total entries (cells): {total_entries}")
print(f"Memory size: {memory_size:.2f} KB")

In [ ]:
import pandas as pd
import openpyxl
from openpyxl.styles import Font

# 1. Loading the original file using openpyxl to preserve tracking layout
wb = openpyxl.load_workbook("Daily Update.xlsx")
ws = wb['Sheet3']

# 2. Extracting and cleaning the existing headers from the first row
raw_headers = [cell.value for cell in ws[1]]
cleaned_headers = [str(h).strip() if h is not None else "" for h in raw_headers]

# 3. Defining the precise font styles requested
header_font = Font(name="Calibri", size=11, bold=True)
body_font = Font(name="Calibri", size=11, bold=False)

# 4. Overwriting row 1 with the cleaned, bolded headers
for col_idx, header_text in enumerate(cleaned_headers, start=1):
    cell = ws.cell(row=1, column=col_idx)
    cell.value = header_text
    cell.font = header_font

# 5. Loop through all remaining rows to enforce Calibri size 11 uniformly
for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
    for cell in row:
        cell.font = body_font

# 6. Save back to a fresh spreadsheet asset
wb.save("Daily_Update_Formatted.xlsx")
print("Success! 'Daily_Update_Formatted.xlsx' has been generated with uniform font and stripped headers.\n")

# 7. Load the newly formatted file into Python SPECIFICALLY for Sheet3
df = pd.read_excel("Daily_Update_Formatted.xlsx", sheet_name='Sheet3')

# Display a clean preview of the first 5 rows
print("=== PREVIEW OF FIRST 5 ROWS ===")
print(df.head(5))

In [ ]:
import pandas as pd
import re
import numpy as np

# 1. Load Sheet3 from your formatted spreadsheet
df = pd.read_excel("Daily_Update_Formatted.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

print("🔄 Starting data correction, column insertion, and cleaning on Sheet3...")

# --- NEW: EXTRACTION OF DAY COLUMN (EXTRACTED BEFORE CLEANING DATE) ---
def extract_day(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().upper()
    # Step A: Check if the day name is explicitly written in the original entry string
    for day in ['MONDAY', 'TUESDAY', 'WEDNESDAY', 'THURSDAY', 'FRIDAY', 'SATURDAY', 'SUNDAY']:
        if day in val_str:
            return day.title()
    # Step B: Fallback calculation if it's already an Excel date object string
    try:
        clean_d = val_str.split(' ')[0]
        dt = pd.to_datetime(clean_d, errors='coerce')
        if pd.notna(dt):
            return dt.strftime('%A')
    except:
        pass
    return np.nan

# Dynamically insert the new 'Day' column directly following 'Apply Date'
if 'Apply Date' in df.columns:
    day_values = df['Apply Date'].apply(extract_day)
    apply_date_idx = df.columns.get_loc('Apply Date')
    df.insert(apply_date_idx + 1, 'Day', day_values)

# --- 1. CLEAN APPLY DATE ---
def clean_date(val):
    if pd.isna(val):
        return val
    val_str = str(val).strip()
    # Remove day suffixes like -MONDAY, -TUESDAY, etc.
    val_cleaned = re.sub(r'[-–—\s]+(MONDAY|TUESDAY|WEDNESDAY|THURSDAY|FRIDAY|SATURDAY|SUNDAY)$', '', val_str, flags=re.IGNORECASE)
    # Remove any lingering timestamp 00:00:00 text
    val_cleaned = val_cleaned.split(' ')[0]
    return val_cleaned

if 'Apply Date' in df.columns:
    df['Apply Date'] = df['Apply Date'].apply(clean_date)

# --- 2. CLEAN SKILL ---
def clean_skill(val):
    if pd.isna(val):
        return val
    s = str(val).strip().upper()
    # Fix structural squished spacing typos
    s = s.replace('SENIORORACLE', 'SENIOR ORACLE')
    s = s.replace('AMAZONREQUIREMENT', 'AMAZON REQUIREMENT')
    return s

if 'Skill' in df.columns:
    df['Skill'] = df['Skill'].apply(clean_skill)

# --- 3. CLEAN NAME ---
if 'Name' in df.columns:
    df['Name'] = df['Name'].astype(str).apply(lambda x: x.strip().title() if x != 'nan' else np.nan)

# --- 4. CLEAN LOCATION ---
def clean_location(val):
    if pd.isna(val):
        return val
    loc = str(val).strip().title()
    # Standardize spelling variations
    loc = loc.replace('Banglore', 'Bangalore')
    loc = loc.replace('Kolkatta', 'Kolkata')
    # If multiple locations exist separated by a comma, pick the primary first hub
    if ',' in loc:
        loc = loc.split(',')[0].strip()
    return loc

if 'Location' in df.columns:
    df['Location'] = df['Location'].apply(clean_location)

# --- 5. CLEAN EXPERIENCE COLUMNS (RELEVANT & TOTAL) ---
def clean_experience(val):
    if pd.isna(val):
        return val
    s = str(val).strip().lower()
    if 'fresh' in s:
        return 0.0
    # Handle specific expressions like '9y 6m' -> 9.5
    if 'y' in s and 'm' in s:
        parts = re.findall(r'\d+\.?\d*', s)
        if len(parts) >= 2:
            return float(parts[0]) + (float(parts[1]) / 12.0)
    # General extraction of numbers
    nums = re.findall(r'\d+\.?\d*', s)
    if nums:
        return float(nums[0])
    return np.nan

for col in ['Relevant Experience', 'Total Experience']:
    if col in df.columns:
        df[col] = df[col].apply(clean_experience)

# --- 6. CLEAN NOTICE PERIOD ---
def clean_notice_period(val):
    if pd.isna(val):
        return val
    s = str(val).strip().lower()
    # Consolidate variants of immediate joiners
    if 'immed' in s or '00:00:00' in s:
        return 'Immediate'
    # Clean numeric shorthand entries like 10D or 20Days to standardized format
    nums = re.findall(r'\d+', s)
    if nums:
        return f"{nums[0]} Days"
    return val

if 'Notice Period' in df.columns:
    df['Notice Period'] = df['Notice Period'].apply(clean_notice_period)

# --- 7. CLEAN CTC COLUMNS (CURRENT & EXPECTED) ---
def clean_ctc(val):
    if pd.isna(val):
        return val
    s = str(val).strip().lower()

    # Remove percentage hike comments completely since they aren't numerical salaries
    if '%' in s or 'hike' in s:
        return np.nan

    # Handle ranges like '9 to 10' or '13-14' by calculating the mathematical average
    if 'to' in s or '-' in s:
        parts = re.findall(r'\d+\.?\d*', s)
        if len(parts) >= 2:
            return (float(parts[0]) + float(parts[1])) / 2.0

    # Standard extraction: remove string markers (LPA, L, lpa) and keep the clean float number
    nums = re.findall(r'\d+\.?\d*', s)
    if nums:
        return float(nums[0])
    return np.nan

for col in ['Current CTC', 'Expected CTC']:
    if col in df.columns:
        df[col] = df[col].apply(clean_ctc)

# Save the beautifully cleaned tracking sheet to a new file asset
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Every column has been cleaned and saved into 'Daily_Update_Cleaned.xlsx'.")

# --- DISPLAY DATASET DIMENSIONS ---
print("\n==========================================================")
print("                  DATASET METRICS & SIZE                  ")
print("==========================================================")
print(f"Total Rows Processed:    {df.shape[0]}")
print(f"Total Columns Maintained: {df.shape[1]} (Includes new 'Day' column)")

# --- DISPLAY CLEAN PREVIEW ---
print("\n==========================================================")
print("         PREVIEW OF COMPLETELY CORRECTED DATA             ")
print("==========================================================")
visible_cols = [c for c in ['Apply Date', 'Day', 'Skill', 'Name', 'Location', 'Relevant Experience', 'Total Experience', 'Notice Period', 'Current CTC', 'Expected CTC'] if c in df.columns]
print(df[visible_cols].head(5))

In [ ]:
import pandas as pd

# 1. Load the cleaned spreadsheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

print("Standardizing dates and removing columns on Sheet3...")

# --- 1. STANDARDIZE APPLY DATE TO DD/MM/YYYY ---
def uniform_date_format(val):
    if pd.isna(val):
        return val

    val_str = str(val).strip()

    try:
        # Handle formats like '2026-03-04' or raw datetime objects
        if '-' in val_str:
            dt = pd.to_datetime(val_str, format='%Y-%m-%d', errors='coerce')
        # Handle formats like '23/12/2025' or '5/02/2026'
        elif '/' in val_str:
            first_part = val_str.split('/')[0]
            if len(first_part) == 4:
                dt = pd.to_datetime(val_str, format='%Y/%m/%d', errors='coerce')
            else:
                dt = pd.to_datetime(val_str, dayfirst=True, errors='coerce')
        else:
            dt = pd.to_datetime(val_str, dayfirst=True, errors='coerce')

        # If successfully parsed, force the clean DD/MM/YYYY string format (with leading zeros)
        if pd.notna(dt):
            return dt.strftime('%d/%m/%Y')
    except:
        pass

    return val_str

if 'Apply Date' in df.columns:
    df['Apply Date'] = df['Apply Date'].apply(uniform_date_format)

# --- 2. REMOVE THE ENTIRE RESPONSE DATE COLUMN ---
if 'Response Date' in df.columns:
    df = df.drop(columns=['Response Date'])
    print("🗑️ 'Response Date' column has been successfully removed.")

# Save the updated sheet back to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("Success! File updated and saved to 'Daily_Update_Cleaned.xlsx'.")

# --- VISUAL PREVIEW ---
print("\n=== PREVIEW OF THE UPDATED COLUMNS ===")
print(df[['Apply Date', 'Day', 'Skill', 'Name']].head(10))

In [ ]:
import pandas as pd

# 1. Load the spreadsheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# Target column name update
phone_col = 'Phone no.'

# Clean up trailing whitespaces for accurate matching
for col in ['Name', phone_col]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

print("==========================================================")
print("             DUPLICATE ENTRY AUDIT REPORT                 ")
print("==========================================================")

# --- CHECK 1: IDENTIFY DUPLICATE NAMES ---
print("\n 1. REPEATED NAMES:")
if 'Name' in df.columns:
    dup_names = df[df.duplicated(subset=['Name'], keep=False)]
    if not dup_names.empty:
        dup_names_sorted = dup_names.sort_values(by='Name')
        print(f" Found {dup_names_sorted.shape[0]} rows with repeating names:")
        for idx, row in dup_names_sorted.iterrows():
            print(f"   - Excel Row {idx + 2}: Name: '{row['Name']}' | Phone: {row.get(phone_col, 'N/A')}")
    else:
        print("   ✅ No duplicate names found.")
else:
    print("   ⚠️ 'Name' column not found in dataset.")


# --- CHECK 2: IDENTIFY DUPLICATE PHONE NUMBERS ---
print(f"\n 2. REPEATED PHONE NUMBERS (Column: '{phone_col}'):")
if phone_col in df.columns:
    # Find all rows where the phone number appears more than once
    dup_phones = df[df.duplicated(subset=[phone_col], keep=False)]

    # Filter out empty entries from being flagged as duplicates
    dup_phones = dup_phones[~dup_phones[phone_col].isin(['nan', '', 'None'])]

    if not dup_phones.empty:
        # Group by phone number so we can see unique lists of where they repeat
        grouped = dup_phones.groupby(phone_col)
        print(f" Found {len(grouped)} distinct repeated phone numbers across {dup_phones.shape[0]} total rows:\n")

        for num, group in grouped:
            # Map index positions to 1-based Excel row numbering numbers
            excel_rows = [str(idx + 2) for idx in group.index]
            names_linked = [f"'{n}'" for n in group['Name'].values if pd.notna(n)]

            print(f"   📱 Number: {num}")
            print(f"      📍 Found at Excel Rows: {', '.join(excel_rows)}")
            print(f"      👤 Associated Profiles: {', '.join(names_linked)}")
            print("      " + "-"*40)
    else:
        print("   ✅ No duplicate phone numbers found.")
else:
    print(f"   ⚠️ '{phone_col}' column not found in dataset. Please check spelling.")


# --- CHECK 3: IDENTIFY ROWS WHERE BOTH MATCH ---
print("\n 3. REPEATED BOTH (SAME NAME & SAME PHONE NUMBER):")
if 'Name' in df.columns and phone_col in df.columns:
    dup_both = df[df.duplicated(subset=['Name', phone_col], keep=False)]
    dup_both = dup_both[~dup_both[phone_col].isin(['nan', '', 'None'])]

    if not dup_both.empty:
        dup_both_sorted = dup_both.sort_values(by='Name')
        print(f" 🎯 Found {dup_both_sorted.shape[0]} exact profile duplicates:")
        for idx, row in dup_both_sorted.iterrows():
            print(f"   - Excel Row {idx + 2}: Name: '{row['Name']}' | Phone: {row[phone_col]}")
    else:
        print("   ✅ No exact matching Name + Phone profiles found.")
else:
    print("   ⚠️ Missing required columns for joint profile check.")

In [ ]:
import pandas as pd

# 1. Load the spreadsheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

print("🔨 Applying updates and re-sequencing 'S.no' column...")

# --- 1. REMOVE EXCEL ROW 5 AND ROW 73 ---
# Excel Row 5  -> Index 3
# Excel Row 73 -> Index 71
indices_to_drop = [3, 71]
df = df.drop(index=[idx for idx in indices_to_drop if idx in df.index])
print("🗑️ Excel Row 5 and Excel Row 73 have been deleted.")

# --- 2. UPDATE RECRUITER NAME FOR BALARAJ (EXCEL ROW 17 -> INDEX 15) ---
recruiter_col = None
for col in df.columns:
    if 'refer' in col.lower() or 'by' in col.lower():
        recruiter_col = col
        break

if recruiter_col and 15 in df.index:
    df.at[15, recruiter_col] = 'Abhilasha & Masharrat'
    print(f"📝 Row 17 '{recruiter_col}' updated to 'Abhilasha & Masharrat'.")

# --- 3. FIX TARGET SERIAL NUMBER COLUMN ('S.no') ---
df = df.reset_index(drop=True)

if 'S.no' in df.columns:
    df['S.no'] = range(1, len(df) + 1)
    print("🔢 'S.no' column has been re-sequenced smoothly without gaps.")
else:
    target_col = [col for col in df.columns if col.lower() == 's.no']
    if target_col:
        df[target_col[0]] = range(1, len(df) + 1)
        print(f"🔢 '{target_col[0]}' column has been re-sequenced smoothly without gaps.")

# 4. Save the cleanly updated tracker sheet back to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("\n✅ Success! File updated and saved to 'Daily_Update_Cleaned.xlsx'.")

# --- NEW: CONSOLE CONFIGURATION FOR FULL ROW PREVIEW ---
# These configurations stop pandas from truncating or wrapping columns text to the next line
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)        # Expand horizontal width allowance
pd.set_option('display.max_colwidth', None) # Don't cut off text inside cells

print("\n==========================================================================")
print("             VERIFICATION PREVIEW: FULL ROW OF MODIFIED BALARAJ           ")
print("==========================================================================")
# Isolate the remaining Balaraj profile and show his entire row matrix layout
balaraj_row = df[df['Name'] == 'Balaraj']
print(balaraj_row)

print(f"\nTotal remaining data rows in dataset: {len(df)}")

In [ ]:
import pandas as pd

# Load the spreadsheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')

# Print only the column names as a clean list
print(list(df.columns))

In [ ]:
import pandas as pd
import numpy as np  # Added to fix the NameError

# Load the spreadsheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

print("📧 Formatting emails and auditing for duplicates...")

# --- 1. CONVERT EMAIL COLUMN TO CLICKABLE LINK FORMAT ---
if 'Email' in df.columns:
    # Clean trailing spaces and normalize to lowercase for consistency
    df['Email'] = df['Email'].astype(str).str.strip().str.lower()

    # Wrap standard email strings into an Excel HYPERLINK formula
    df['Email'] = df['Email'].apply(
        lambda x: f'=HYPERLINK("mailto:{x}", "{x}")' if x not in ['nan', '', 'none'] else np.nan
    )
    print("🔗 All valid email entries have been converted into active hyperlink formulas.")
else:
    print("⚠️ 'Email' column not found in the dataset.")

# --- 2. IDENTIFY AND SHOW DUPLICATE EMAIL ENTRIES ---
print("\n==========================================================")
print("             DUPLICATE EMAIL AUDIT REPORT                 ")
print("==========================================================")

if 'Email' in df.columns:
    # Find all rows with repeating email formulas
    dup_emails = df[df.duplicated(subset=['Email'], keep=False)]

    # Ignore empty rows from triggering duplicate warnings
    dup_emails = dup_emails[~dup_emails['Email'].isna()]

    if not dup_emails.empty:
        # Group duplicates together to cleanly list their exact row numbers
        grouped = dup_emails.groupby('Email')
        print(f" Found {len(grouped)} distinct repeated email addresses:\n")

        for email_formula, group in grouped:
            # Extract the raw email text out of the formula wrapper for display
            raw_email = email_formula.split('"')[3]

            # Map index positions to 1-based Excel row numbers
            excel_rows = [str(idx + 2) for idx in group.index]
            names_linked = [f"'{name}'" for name in group['Name'].values if pd.notna(name)]

            print(f"   📧 Email: {raw_email}")
            print(f"      📍 Found at Excel Rows: {', '.join(excel_rows)}")
            print(f"      👤 Associated Profiles: {', '.join(names_linked)}")
            print("      " + "-"*40)
    else:
        print("   ✅ No duplicate email entries found.")

# Save the updated sheet back to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("\n✅ Success! File updated and saved to 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd

# Load the spreadsheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

print(" Analyzing 'Location' column footprint...")

if 'Location' in df.columns:
    # --- 1. CLEAN UP TEXT INCONSISTENCIES ---
    # Convert 'nan' entries to an official blank marker
    df['Location'] = df['Location'].replace(['nan', 'None', 'none'], pd.NA)

    # Strip hidden whitespaces and convert to Title Case (e.g., 'bangalore ' -> 'Bangalore')
    df['Location'] = df['Location'].astype(str).str.strip().str.title()

    # Put back actual NA markers for values that were missing
    df['Location'] = df['Location'].replace(['<Na>', 'Nan', 'None'], pd.NA)

    # --- 2. CALCULATE METRICS ---
    # Count only valid, non-null distinct entries
    unique_locations_count = df['Location'].dropna().nunique()

    # Generate the volume breakdown for each location
    location_counts = df['Location'].dropna().value_counts()

    # Calculate how many rows are missing a location
    missing_count = df['Location'].isna().sum()

    # --- 3. PRINT REPORT ---
    print("\n==========================================================")
    print("                LOCATION PROFILE REPORT                   ")
    print("==========================================================")
    print(f" Total Distinct Locations Found: {unique_locations_count}")
    print("----------------------------------------------------------")
    print(f"{'City / Location':<30} | {'Candidate Volume':<15}")
    print("----------------------------------------------------------")

    for city, count in location_counts.items():
        print(f"{city:<30} | {count:<15} candidates")

    if missing_count > 0:
        print("----------------------------------------------------------")
        print(f"{'Unspecified / Blank':<30} | {missing_count:<15} rows")

else:
    print("⚠️ 'Location' column not found in the dataset. Please verify the spelling.")

In [ ]:
import pandas as pd

# Load the spreadsheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

print(" Standardizing locations to clean city names...")

if 'Location' in df.columns:
    # 1. Baseline Text Normalization (strip spaces and set to Title Case)
    df['Location'] = df['Location'].astype(str).str.strip().str.title()

    # 2. Apply Custom Standardization Rules
    mapping_rules = {
        'Odisha': 'Not Informed by Candidate',
        'Chattisgarh': 'Not Informed by Candidate',
        'Pune Tirupathi': 'Not Informed by Candidate',
        'Tamil Nadu Koimbatur': 'Coimbatore',
        'Nan': 'Not Informed by Candidate',
        'None': 'Not Informed by Candidate'
    }

    # Map the targeted messy strings to their clean equivalents
    df['Location'] = df['Location'].replace(mapping_rules)

    # Safely handle any remaining raw blanks or null placeholders
    df['Location'] = df['Location'].fillna('Not Informed by Candidate')
    df.loc[df['Location'] == '<Na>', 'Location'] = 'Not Informed by Candidate'

    # 3. Calculate Updated Summary Metrics
    # Filter out 'Not Informed by Candidate' to get the true city count
    true_cities = df[df['Location'] != 'Not Informed by Candidate']['Location']
    unique_locations_count = true_cities.nunique()

    # Generate the value counts for all entries
    location_counts = df['Location'].value_counts()

    # --- 4. PRINT REPORT ---
    print("\n==========================================================")
    print("           UPDATED LOCATION PROFILE REPORT                ")
    print("==========================================================")
    print(f" Total Distinct Valid Cities Found: {unique_locations_count}")
    print("----------------------------------------------------------")
    print(f"{'City / Location':<30} | {'Candidate Volume':<15}")
    print("----------------------------------------------------------")

    # Print valid cities first
    for city, count in location_counts.items():
        if city != 'Not Informed by Candidate':
            print(f"{city:<30} | {count:<15} candidates")

    print("----------------------------------------------------------")
    # Print the uninformative/blank bucket at the bottom for clean data hygiene
    if 'Not Informed by Candidate' in location_counts:
        print(f"{'Not Informed by Candidate':<30} | {location_counts['Not Informed by Candidate']:<15} rows")

else:
    print("⚠️ 'Location' column not found in the dataset.")

# Save the cleanly updated tracker sheet back to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("\n Success! File updated and saved to 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load both excel sheets securely
df_clean = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')

# Clean leading/trailing spaces from column names
df_clean.columns = [col.strip() for col in df_clean.columns]
df_orig.columns = [col.strip() for col in df_orig.columns]

print(" Extracting raw emails and injecting active hyperlinks...")

# 2. Create standardized lower-case matching links to prevent mismatches
df_clean['Match_Key'] = df_clean['Name'].astype(str).str.strip().str.lower()
df_orig['Match_Key'] = df_orig['Name'].astype(str).str.strip().str.lower()

# Create lookup dictionary: {Candidate Name: Raw Email}
email_lookup = dict(zip(df_orig['Match_Key'], df_orig['Email']))

def format_to_hyperlink(key):
    raw_email = email_lookup.get(key)
    # Check for empty cells or missing data entries
    if pd.isna(raw_email) or str(raw_email).strip().lower() in ['nan', 'none', '']:
        return np.nan

    clean_email = str(raw_email).strip().lower()
    return f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'

# 3. Apply the formatting to the Email column only
df_clean['Email'] = df_clean['Match_Key'].apply(format_to_hyperlink)

# Drop helper tracking keys
df_clean.drop(columns=['Match_Key'], errors='ignore', inplace=True)

# 4. Save modifications back directly over your file
df_clean.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! 'Email' column entries restored and saved to 'Daily_Update_Cleaned.xlsx'.")

# --- INSTANT CONSOLE INTEGRITY CHECK ---
print("\n==========================================================")
print("             EMAIL UPDATE RESTORATION AUDIT               ")
print("==========================================================")
null_count = df_clean['Email'].isna().sum()
total_rows = len(df_clean)
active_links = total_rows - null_count

print(f" Total Candidate Rows: {total_rows}")
print(f" Active Clickable Hyperlinks Inserted: {active_links}")
print(f" Rows without an email entry: {null_count}")

print("\n Top 5 Row Sample Preview:")
print(df_clean[['Name', 'Email']].head(5))

In [ ]:
import pandas as pd

# Load the spreadsheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

print(" Analyzing 'Remark' column footprints...")

if 'Remark' in df.columns:
    # --- 1. CLEAN UP TEXT INCONSISTENCIES ---
    # Handle missing/empty cells by filling them with a standard placeholder string
    df['Remark'] = df['Remark'].fillna('Blank / No Remark')

    # Strip spaces and convert common string variants to a clean format
    df['Remark'] = df['Remark'].astype(str).str.strip()
    df['Remark'] = df['Remark'].replace(['nan', 'None', 'none', ''], 'Blank / No Remark')

    # --- 2. CALCULATE METRICS ---
    # Count unique remarks (excluding the placeholder bucket for an accurate count)
    true_remarks = df[df['Remark'] != 'Blank / No Remark']['Remark']
    unique_remarks_count = true_remarks.nunique()

    # Generate the volume breakdown for each unique remark phrase
    remark_counts = df['Remark'].value_counts()

    # --- 3. PRINT REPORT ---
    print("\n==========================================================")
    print("                 REMARK BREAKDOWN REPORT                  ")
    print("==========================================================")
    print(f" Total Distinct Feedback Remarks Found: {unique_remarks_count}")
    print("----------------------------------------------------------")
    print(f"{'Candidate Status / Remark Phrase':<45} | {'Count':<10}")
    print("----------------------------------------------------------")

    # Print real remarks first
    for remark, count in remark_counts.items():
        if remark != 'Blank / No Remark':
            print(f"{remark:<45} | {count:<10} entries")

    print("----------------------------------------------------------")
    # Print the blank tracking bucket at the bottom
    if 'Blank / No Remark' in remark_counts:
        print(f"{'Blank / No Remark':<45} | {remark_counts['Blank / No Remark']:<10} rows")

else:
    print("⚠️ 'Remark' column not found in the dataset. Please verify the column name.")

In [ ]:
import pandas as pd

# Load the spreadsheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

print(" Analyzing 'Remark' column footprints...")

if 'Remark' in df.columns:
    # --- 1. CLEAN UP TEXT INCONSISTENCIES ---
    # Handle missing/empty cells by filling them with a standard placeholder string
    df['Remark'] = df['Remark'].fillna('Blank / No Remark')

    # Strip spaces and convert common string variants to a clean format
    df['Remark'] = df['Remark'].astype(str).str.strip()
    df['Remark'] = df['Remark'].replace(['nan', 'None', 'none', ''], 'Blank / No Remark')

    # --- 2. CALCULATE METRICS ---
    # Count unique remarks (excluding the placeholder bucket for an accurate count)
    true_remarks = df[df['Remark'] != 'Blank / No Remark']['Remark']
    unique_remarks_count = true_remarks.nunique()

    # Generate the volume breakdown for each unique remark phrase
    remark_counts = df['Remark'].value_counts()

    # --- 3. PRINT REPORT ---
    print("\n==========================================================")
    print("                 REMARK BREAKDOWN REPORT                  ")
    print("==========================================================")
    print(f" Total Distinct Feedback Remarks Found: {unique_remarks_count}")
    print("----------------------------------------------------------")
    print(f"{'Candidate Status / Remark Phrase':<45} | {'Count':<10}")
    print("----------------------------------------------------------")

    # Print real remarks first
    for remark, count in remark_counts.items():
        if remark != 'Blank / No Remark':
            print(f"{remark:<45} | {count:<10} entries")

    print("----------------------------------------------------------")
    # Print the blank tracking bucket at the bottom
    if 'Blank / No Remark' in remark_counts:
        print(f"{'Blank / No Remark':<45} | {remark_counts['Blank / No Remark']:<10} rows")

else:
    print("⚠️ 'Remark' column not found in the dataset. Please verify the column name.")

In [ ]:
import pandas as pd
import numpy as np
import re

# 1. Load the spreadsheet you are working on
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. Secure raw emails from the source sheet to protect them from disappearing
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a quick key-value map for the emails: {lowercase name: email}
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

print(" Parsing and transforming 'Remark' column into clean data dimensions...")

# Initialize the new structured columns with safe default values
df['Profile Status'] = 'Not Noted'
df['Client Name'] = 'N/A'
df['Client Share Date'] = 'N/A'
df['Assigned Recruiter Follow-up'] = 'N/A'
df['Client Database Duplicate'] = 'No'

# Dictionary mapping text patterns to standard Recruiter names
recruiter_keywords = {
    'abhilasha': 'Abhilasha',
    'sharf': 'Sharf',
    'masarrat': 'Masarrat',
    'neha': 'Neha',
    'sharf & neha': 'Sharf & Neha'
}

# --- PROCESS AND PARSE ROW BY ROW ---
for idx, row in df.iterrows():
    # --- PROTECT AND RE-APPLY EMAIL HYPERLINK ---
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = np.nan

    # --- REMARK PARSING LOGIC ---
    remark = str(row.get('Remark', '')).strip()

    # Check for empty, missing, or blank tracking entries
    if not remark or remark.lower() in ['nan', 'none', 'blank / no remark', '']:
        continue  # Keeps the default 'Not Noted' state set up above

    # 1. Handle Hard Rejects
    if remark.lower() == 'reject':
        df.at[idx, 'Profile Status'] = 'Internal Reject'
        continue

    # 2. Handle Client Database Duplicates
    if 'already exist in client data base' in remark.lower() or 'cannot be considered' in remark.lower():
        df.at[idx, 'Profile Status'] = 'Rejected by Client'
        df.at[idx, 'Client Database Duplicate'] = 'Yes'
        # Try to capture client context if mentioned in the text snippet
        if 'infosys' in remark.lower():
            df.at[idx, 'Client Name'] = 'Infosys'
        continue

    # 3. Handle Active Shared Profiles
    if 'shared to' in remark.lower():
        df.at[idx, 'Profile Status'] = 'Shared'

        # Extract Client Name
        if 'infosys' in remark.lower():
            df.at[idx, 'Client Name'] = 'Infosys'
        elif 'sidra' in remark.lower() or 'force' in remark.lower():
            df.at[idx, 'Client Name'] = 'Sidra Force'
        elif 'westernacher' in remark.lower():
            df.at[idx, 'Client Name'] = 'Westernacher'

        # Extract Recruiter follow up ownership
        if 'sharf & neha' in remark.lower():
            df.at[idx, 'Assigned Recruiter Follow-up'] = 'Sharf & Neha'
        else:
            for keyword, clean_name in recruiter_keywords.items():
                if keyword in remark.lower():
                    df.at[idx, 'Assigned Recruiter Follow-up'] = clean_name
                    break

        # Extract the hidden sharing Date using a regular expression match
        date_match = re.search(r'(\d{1,2})\s*([a-zA-Z]{3})|([a-zA-Z]{3})\s*(\d{1,2})', remark)
        if date_match:
            day = date_match.group(1) or date_match.group(4)
            month = date_match.group(2) or date_match.group(3)
            # Normalize day text padding (e.g., "5" -> "05")
            day = f"0{day.strip()}" if len(day.strip()) == 1 else day.strip()
            # Standardize and capitalize monthly format
            month = month.strip().lower().capitalize()
            df.at[idx, 'Client Share Date'] = f"{day} {month}"

# --- DROP THE OLD UNSTRUCTURED REMARK COLUMN ---
if 'Remark' in df.columns:
    df = df.drop(columns=['Remark'])
    print(" Old 'Remark' text column removed successfully.")

# Reorder columns dynamically to place new dimensions neatly right before 'Update From Client Side'
cols = list(df.columns)
target_idx = cols.index('Update From Client Side') if 'Update From Client Side' in cols else len(cols)
new_features = ['Profile Status', 'Client Name', 'Client Share Date', 'Assigned Recruiter Follow-up', 'Client Database Duplicate']

# Pull new features out of the end of the list and splice them back in position
for f in new_features:
    if f in cols: cols.remove(f)
for i, f in enumerate(new_features):
    cols.insert(target_idx + i, f)
df = df[cols]

# Save file modifications
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! File updated and saved to 'Daily_Update_Cleaned.xlsx'.")

# --- DISPLAY STRUCTURAL VALIDATION PREVIEW ---
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print("\n==========================================================================================")
print("                    DATA PIPELINE SPLIT PREVIEW (LAST 8 ROWS)                             ")
print("==========================================================================================")
print(df[['Name', 'Email', 'Profile Status', 'Client Name', 'Client Share Date', 'Assigned Recruiter Follow-up', 'Client Database Duplicate']].tail(8))

In [ ]:
import pandas as pd

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

print("🔍 Auditing dataset for missing Client Share Dates...")

# 2. Define the filtering criteria:
# Has a valid Client Name (not blank, not N/A) BUT has an empty or N/A Share Date
has_client = df['Client Name'].notna() & (df['Client Name'] != 'N/A')
missing_date = df['Client Share Date'].isna() | (df['Client Share Date'] == 'N/A')

# Combine criteria to find the targets
lazy_hr_df = df[has_client & missing_date]

# 3. Print the results directly to your console
print("\n==========================================================================================")
print(f"🚨 FOUND {len(lazy_hr_df)} ROWS WITH A MENTIONED CLIENT BUT NO SHARE DATE:")
print("==========================================================================================")

if not lazy_hr_df.empty:
    # Display the specific target columns for quick manual review
    columns_to_show = ['S.no', 'Name', 'Client Name', 'Client Share Date', 'Assigned Recruiter Follow-up']
    print(lazy_hr_df[columns_to_show].to_string(index=False))
else:
    print("✅ Perfect! Every profile with a client name has an associated share date listed.")
print("==========================================================================================")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 🛡️ EMAIL PROTECTION LAYER: Pull raw email values from the source to prevent formula wipeout
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create an unbreakable email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying dates
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = np.nan

print(" Updating missing Client Share Dates to 'Not Noted by HR'...")

# 2. Define the exact flag conditions
has_client = df['Client Name'].notna() & (df['Client Name'] != 'N/A')
missing_date = df['Client Share Date'].isna() | (df['Client Share Date'] == 'N/A')

# 3. Apply the 'Not Noted by HR' string value only to the identified records
df.loc[has_client & missing_date, 'Client Share Date'] = 'Not Noted by HR'

# 4. Save your modifications back to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

# --- AUTOMATED CONSOLE AUDIT PREVIEW ---
print("\n==========================================================================================")
print("                       POST-UPDATE DATA INTEGRITY REVIEW                                  ")
print("==========================================================================================")
updated_rows = df[df['Client Share Date'] == 'Not Noted by HR']
if not updated_rows.empty:
    print(updated_rows[['S.no', 'Name', 'Email', 'Client Name', 'Client Share Date']].to_string(index=False))
else:
    print("❌ Error: No matching rows found or updated.")
print("==========================================================================================")

In [ ]:
import pandas as pd

# 1. Load your master tracking sheet in read-only mode
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

print(" Fetching unique entries from 'Update From Client Side'...")

if 'Update From Client Side' in df.columns:
    # 2. Count occurrences of each distinct value (including blanks)
    value_counts = df['Update From Client Side'].value_counts(dropna=False)

    print("\n==========================================================")
    print(f"{'Distinct Client Status / Entry':<35} | {'Occurrences (Count)':<15}")
    print("==========================================================")

    for status, count in value_counts.items():
        # Label the empty cells clearly on the screen output
        if pd.isna(status):
            display_status = "⚠️ [Blank / Empty Cell]"
        else:
            display_status = str(status).strip()

        print(f"{display_status:<35} | {count:<15}")

    print("==========================================================")
    print(f" Total overall rows scanned: {len(df)}")
else:
    print("❌ Error: Could not find a column named 'Update From Client Side' in this sheet.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to prevent formula wipeout
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying columns
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = np.nan

print(" Transforming and splitting columns into 'Update From Client' & 'Remark From Client'...")

# 3. Initialize the two new target columns
df['Update From Client'] = 'Not Noted by HR'
df['Remark From Client'] = 'Not Noted by HR'

# 4. Map the old column values row-by-row using your precise rules
for idx, row in df.iterrows():
    val = str(row.get('Update From Client Side', '')).strip()

    # Handle Blanks
    if not val or val.lower() in ['nan', 'none', '']:
        df.at[idx, 'Update From Client'] = 'Not Noted by HR'
        df.at[idx, 'Remark From Client'] = 'Not Noted by HR'

    # Handle Not Selected
    elif val == 'Not Selected':
        df.at[idx, 'Update From Client'] = 'Not Selected'
        df.at[idx, 'Remark From Client'] = 'Not Selected'

    # Handle Reject
    elif val == 'Reject':
        df.at[idx, 'Update From Client'] = 'Rejected'
        df.at[idx, 'Remark From Client'] = 'Rejected'

    # Handle Duplicate Candidate
    elif 'duplicate' in val.lower():
        df.at[idx, 'Update From Client'] = 'Duplicate Profile Entry'
        df.at[idx, 'Remark From Client'] = 'Duplicate Profile Entry'

    # Handle Interview/Certificates Case
    elif 'conducted interview' in val.lower():
        df.at[idx, 'Update From Client'] = 'Reached Interview Phase'
        df.at[idx, 'Remark From Client'] = 'Provision of Certificates Rejected by Candidate'

    # Fallback/Safety clause
    else:
        df.at[idx, 'Update From Client'] = val
        df.at[idx, 'Remark From Client'] = val

# 5. Drop the old unstructured column safely
if 'Update From Client Side' in df.columns:
    df = df.drop(columns=['Update From Client Side'])
    print(" Old 'Update From Client Side' column removed successfully.")

# Reorder columns dynamically to append the two new features at the end of the sheet
cols = list(df.columns)
new_features = ['Update From Client', 'Remark From Client']
for f in new_features:
    if f in cols: cols.remove(f)
cols.extend(new_features)
df = df[cols]

# 6. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet columns successfully generated and saved over 'Daily_Update_Cleaned.xlsx'.")

# --- CONSOLE VALIDATION PREVIEW ---
print("\n==========================================================================================")
print("                       POST-SPLIT COLUMN INTEGRITY AUDIT                                  ")
print("==========================================================================================")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(df[['Name', 'Email', 'Update From Client', 'Remark From Client']].head(10))
print("==========================================================================================")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to prevent formula wipeout
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = np.nan

print(" Updating 'Client Database Duplicate' for S.no 62...")

# 3. Target exactly where S.no is 62 and update 'Client Database Duplicate' to 'Yes'
if 'Client Database Duplicate' in df.columns:
    df.loc[df['S.no'] == 62, 'Client Database Duplicate'] = 'Yes'
else:
    print("❌ Error: 'Client Database Duplicate' column not found.")

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

# --- CONSOLE VALIDATION PREVIEW ---
print("\n==========================================================================================")
print("                           POST-UPDATE DATA INTEGRITY AUDIT                               ")
print("==========================================================================================")
target_row = df[df['S.no'] == 62]
print(target_row[['S.no', 'Name', 'Email', 'Client Database Duplicate']])
print("==========================================================================================")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to prevent formula wipeout
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = np.nan

print(" Modifying 'Not Noted by HR' entries to 'Not Selected'...")

# 3. Target both columns and replace the specific text values
columns_to_update = ['Update From Client', 'Remark From Client']

for col in columns_to_update:
    if col in df.columns:
        df[col] = df[col].replace({'Not Noted by HR': 'Not Selected'})
    else:
        print(f"⚠️ Warning: Column '{col}' not found in the sheet.")

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

# --- CONSOLE VALIDATION PREVIEW ---
print("\n==========================================================================================")
print("                           POST-UPDATE VALUE DISTRIBUTION                                 ")
print("==========================================================================================")
for col in columns_to_update:
    if col in df.columns:
        print(f"\n📊 Value distribution for column '{col}':")
        print(df[col].value_counts().to_string())
print("==========================================================================================")

In [ ]:
import pandas as pd

# 1. Load your master tracking sheet in a safe, read-only session
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

print(" Auditing the 'Phone no.' column for duplicate entries...")

if 'Phone no.' in df.columns:
    # Remove any empty or missing phone numbers from the duplicate check
    valid_phones = df[df['Phone no.'].notna() & (df['Phone no.'].astype(str).str.strip() != '')].copy()

    # Strip any accidental white spaces from the phone number values to ensure a precise match
    valid_phones['Cleaned_Phone'] = valid_phones['Phone no.'].astype(str).str.strip()

    # Identify all rows that have repeating phone numbers
    duplicates_df = valid_phones[valid_phones.duplicated(subset=['Cleaned_Phone'], keep=False)]

    print("\n==========================================================================================")
    print(f" DUPLICATE PHONE NUMBER AUDIT REPORT (Found {len(duplicates_df)} overlapping entries):")
    print("==========================================================================================")

    if not duplicates_df.empty:
        # Sort by phone number so repeating rows sit right next to each other
        sorted_duplicates = duplicates_df.sort_values(by='Cleaned_Phone')
        print(sorted_duplicates[['S.no', 'Name', 'Phone no.']].to_string(index=False))
    else:
        print(" Clean Record! Every single phone number entry in your spreadsheet is completely unique.")
    print("==========================================================================================")
else:
    print("❌ Error: Could not find a column named 'Phone no.' in this sheet.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to prevent formula wipeout
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = np.nan

print(" Filling blank phone numbers with 'Not Noted by HR'...")

# 3. Target the 'Phone no.' column and replace completely empty/NaN values
if 'Phone no.' in df.columns:
    # First convert to string but replace any text 'nan' or empty spacing with 'Not Noted by HR'
    df['Phone no.'] = df['Phone no.'].fillna('Not Noted by HR')
    df['Phone no.'] = df['Phone no.'].apply(lambda x: 'Not Noted by HR' if str(x).strip() in ['', 'nan', 'NaN', 'None'] else x)
else:
    print("❌ Error: 'Phone no.' column not found in the sheet.")

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

# --- CONSOLE VALIDATION PREVIEW ---
print("\n==========================================================================================")
print("                           POST-UPDATE DATA INTEGRITY AUDIT                               ")
print("==========================================================================================")
not_noted_count = (df['Phone no.'] == 'Not Noted by HR').sum()
print(f" Total Rows where Phone is 'Not Noted by HR': {not_noted_count}")
print("\n Top 5 rows sample preview:")
print(df[['S.no', 'Name', 'Phone no.', 'Email']].head(5))
print("==========================================================================================")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

print(" Processing empty email fields and locking existing hyperlinks...")

# 3. Check and apply changes row-by-row
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    # If a real email address exists in the source file, write it as a secure hyperlink formula
    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        # If it is blank or missing, overwrite explicitly with your requested text
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print("Standardizing columns ('Refered By', 'Location', 'Relevant Experience', 'Total Experience')...")

# 3. Clean 'Refered By' column
if 'Refered By' in df.columns:
    df['Refered By'] = df['Refered By'].fillna('Not Noted by HR')
    df['Refered By'] = df['Refered By'].apply(lambda x: 'Not Noted by HR' if str(x).strip() in ['', 'nan', 'NaN', 'None'] else x)

# Helper clean function to handle case-insensitive text replacements and fill empty fields safely
def clean_candidate_info(value):
    val_str = str(value).strip().lower()
    if not value or val_str in ['', 'nan', 'nan', 'none', 'blank']:
        return 'Not Provided by Candidate'
    # Catches spelling variants like 'not informed by candidate' or 'not informed by by candidate'
    if 'not informed' in val_str:
        return 'Not Provided by Candidate'
    return value

# 4. Apply transformations to 'Location', 'Relevant Experience', and 'Total Experience' columns
target_cols = ['Location', 'Relevant Experience', 'Total Experience']

for col in target_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_candidate_info)
    else:
        print(f"⚠️ Warning: Column '{col}' not found in the sheet.")

# 5. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

# --- CONSOLE VALIDATION PREVIEW ---
print("\n==========================================================================================")
print("                           POST-UPDATE DATA INTEGRITY AUDIT                               ")
print("==========================================================================================")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(df[['Name', 'Refered By', 'Location', 'Relevant Experience', 'Total Experience']].head(10))
print("==========================================================================================")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Standardizing missing values in 'Notice Period', 'Current CTC', and 'Expected CTC'...")

# Helper function to catch all structural variants of blank/empty inputs
def clean_compensation_info(value):
    val_str = str(value).strip().lower()
    if not value or val_str in ['', 'nan', 'none', 'blank', 'null']:
        return 'Not Provided by Candidate'
    return value

# 3. Apply transformations to the targeted financial/timeline columns
target_cols = ['Notice Period', 'Current CTC', 'Expected CTC']

for col in target_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_compensation_info)
    else:
        print(f"⚠️ Warning: Column '{col}' not found in the sheet.")

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Modifying 'Not Noted' entries to 'Not Noted by HR' in Profile Status...")

# 3. Target the 'Profile Status' column and replace the specific text values
if 'Profile Status' in df.columns:
    df['Profile Status'] = df['Profile Status'].replace({'Not Noted': 'Not Noted by HR'})
else:
    print("⚠️ Warning: Column 'Profile Status' not found in the sheet.")

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Modifying 'Not Noted by HR' entries to 'Internal Reject' in Profile Status...")

# 3. Target the 'Profile Status' column and replace the specific text values
if 'Profile Status' in df.columns:
    df['Profile Status'] = df['Profile Status'].replace({'Not Noted by HR': 'Internal Reject'})
else:
    print("⚠️ Warning: Column 'Profile Status' not found in the sheet.")

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Syncing 'Client Name' values with 'Profile Status' rules...")

# 3. Apply conditional logical overwrites based on Profile Status
if 'Profile Status' in df.columns and 'Client Name' in df.columns:
    # Rule A: Wherever there's 'Internal Reject' in profile status, put 'Internal Reject' in client name
    df.loc[df['Profile Status'] == 'Internal Reject', 'Client Name'] = 'Internal Reject'

    # Rule B: If there's 'Rejected by Client' in profile status, put 'Rejected by Client' in client name
    df.loc[df['Profile Status'] == 'Rejected by Client', 'Client Name'] = 'Rejected by Client'
else:
    print("⚠️ Error: 'Profile Status' or 'Client Name' column missing from sheet.")

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

# --- CONSOLE VALIDATION PREVIEW ---
print("\n==========================================================================================")
print("                           POST-UPDATE VALUE DISTRIBUTION                                 ")
print("==========================================================================================")
if 'Client Name' in df.columns:
    print(" Value distribution for updated column 'Client Name':")
    print(df['Client Name'].value_counts().to_string())
print("==========================================================================================")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Updating 'Profile Status' to 'Rejected by Client (Duplicate)' based on duplicate records...")

# 3. Target rows where 'Client Database Duplicate' is exactly 'Yes'
if 'Client Database Duplicate' in df.columns and 'Profile Status' in df.columns:
    df.loc[df['Client Database Duplicate'] == 'Yes', 'Profile Status'] = 'Rejected by Client (Duplicate)'
else:
    print("⚠️ Error: 'Client Database Duplicate' or 'Profile Status' column missing from sheet.")

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Syncing 'Client Name' values with updated 'Profile Status' rules...")

# 3. Apply conditional logical overwrites based on Profile Status
if 'Profile Status' in df.columns and 'Client Name' in df.columns:
    # Rule A: Wherever there's 'Rejected by Client (Duplicate)' in profile status, put 'N/A (Duplicate)'
    df.loc[df['Profile Status'] == 'Rejected by Client (Duplicate)', 'Client Name'] = 'N/A (Duplicate)'

    # Rule B: Wherever there's 'Internal Reject' in profile status, put 'N/A (Internal Reject)'
    df.loc[df['Profile Status'] == 'Internal Reject', 'Client Name'] = 'N/A (Internal Reject)'
else:
    print("⚠️ Error: 'Profile Status' or 'Client Name' column missing from sheet.")

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Processing 'Client Share Date' sync and date conversions...")

# 3. Apply the conditional rules and date formatting updates row-by-row
if 'Client Name' in df.columns and 'Client Share Date' in df.columns:

    for idx, row in df.iterrows():
        client_name_val = str(row['Client Name']).strip()
        share_date_val = row['Client Share Date']

        # Rule A & B: Map status tags from Client Name to Client Share Date
        if client_name_val == 'N/A (Internal Reject)':
            df.at[idx, 'Client Share Date'] = 'N/A (Internal Reject)'
        elif client_name_val == 'N/A (Duplicate)':
            df.at[idx, 'Client Share Date'] = 'N/A (Duplicate)'

        # Rule C: Convert valid dates to standard dd/mm/yyyy string format
        else:
            if pd.notna(share_date_val) and str(share_date_val).strip().lower() not in ['nan', 'none', '']:
                try:
                    # Parse into a datetime object and output cleanly
                    parsed_date = pd.to_datetime(share_date_val)
                    df.at[idx, 'Client Share Date'] = parsed_date.strftime('%d/%m/%Y')
                except (ValueError, TypeError):
                    # Keep original value if it's already an unparseable custom text flag
                    pass
else:
    print("⚠️ Error: 'Client Name' or 'Client Share Date' column missing from sheet.")

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

# --- CONSOLE VALIDATION PREVIEW ---
print("\n==========================================================================================")
print("                           POST-UPDATE DATA INTEGRITY AUDIT                               ")
print("==========================================================================================")
if 'Client Share Date' in df.columns:
    print("📊 Sample breakdown for updated 'Client Share Date' column:")
    print(df['Client Share Date'].value_counts().head(10).to_string())
print("==========================================================================================")

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Detecting sheet baseline timeline and standardizing short dates...")

# 3. Determine the most frequent year across the 'Apply Date' column to use as a fallback default year
default_year = 2025  # Standard fallback
if 'Apply Date' in df.columns:
    years = []
    for val in df['Apply Date'].dropna():
        year_match = re.search(r'\b(202\d)\b', str(val))
        if year_match:
            years.append(int(year_match.group(1)))
        else:
            try:
                years.append(pd.to_datetime(val).year)
            except:
                pass
    if years:
        default_year = max(set(years), key=years.count)

# 4. Standardize text dates like '30 Dec' or '23 Jan' row-by-row
if 'Client Share Date' in df.columns:
    for idx, row in df.iterrows():
        share_date_val = row['Client Share Date']

        # Skip rows that contain status labels or are completely empty
        if pd.isna(share_date_val) or str(share_date_val).strip() in ['', 'nan', 'NaN', 'None', 'Not Noted by HR', 'N/A (Internal Reject)', 'N/A (Duplicate)']:
            continue

        date_str = str(share_date_val).strip()

        # Check if the text matches patterns like "30 Dec" or "23 Jan"
        match = re.match(r'^(\d{1,2})\s+([A-Za-z]{3,})$', date_str)
        if match:
            day = int(match.group(1))
            month_str = match.group(2)[:3].title()

            # Dynamically pull the exact year for this row from 'Apply Date' if available
            row_year = default_year
            if 'Apply Date' in df.columns and pd.notna(row['Apply Date']):
                apply_str = str(row['Apply Date'])
                year_match = re.search(r'\b(202\d)\b', apply_str)
                if year_match:
                    row_year = int(year_match.group(1))
                else:
                    try:
                        row_year = pd.to_datetime(row['Apply Date']).year
                    except:
                        pass

            try:
                dt = datetime.strptime(f"{day} {month_str} {row_year}", "%d %b %Y")
                df.at[idx, 'Client Share Date'] = dt.strftime('%d/%m/%Y')
            except:
                pass
        else:
            # If it's already an unformatted Timestamp or another string format, convert it cleanly
            try:
                parsed_date = pd.to_datetime(share_date_val)
                df.at[idx, 'Client Share Date'] = parsed_date.strftime('%d/%m/%Y')
            except:
                pass
else:
    print("⚠️ Error: 'Client Share Date' column missing from sheet.")

# 5. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! All short date formats updated and saved over 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Processing 'Assigned Recruiter Follow-up' and 'Client Name' conditional updates...")

# 3. Rule 1: Map from 'Client Share Date' over to 'Assigned Recruiter Follow-up'
if 'Client Share Date' in df.columns and 'Assigned Recruiter Follow-up' in df.columns:
    for idx, row in df.iterrows():
        share_date_val = str(row['Client Share Date']).strip()
        if share_date_val in ['N/A (Duplicate)', 'N/A (Internal Reject)']:
            df.at[idx, 'Assigned Recruiter Follow-up'] = share_date_val
else:
    print("⚠️ Warning: 'Client Share Date' or 'Assigned Recruiter Follow-up' column missing from sheet.")

# 4. Rule 2: Wherever there's 'Internal Reject' in profile status, put 'N/A (Internal Reject)' in Client Name
if 'Profile Status' in df.columns and 'Client Name' in df.columns:
    df.loc[df['Profile Status'] == 'Internal Reject', 'Client Name'] = 'N/A (Internal Reject)'
else:
    print("⚠️ Warning: 'Profile Status' or 'Client Name' column missing from sheet.")

# 5. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Filling empty cells in 'Assigned Recruiter Follow-up' with 'Not Noted by HR'...")

# 3. Target the 'Assigned Recruiter Follow-up' column and fill empty/NaN records
if 'Assigned Recruiter Follow-up' in df.columns:
    df['Assigned Recruiter Follow-up'] = df['Assigned Recruiter Follow-up'].fillna('Not Noted by HR')
    df['Assigned Recruiter Follow-up'] = df['Assigned Recruiter Follow-up'].apply(
        lambda x: 'Not Noted by HR' if str(x).strip() in ['', 'nan', 'NaN', 'None', 'null'] else x
    )
else:
    print("❌ Error: 'Assigned Recruiter Follow-up' column not found in the sheet.")

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Updating certificate remarks in 'Remark From Client' column...")

# 3. Target the 'Remark From Client' column and replace the specified text
if 'Remark From Client' in df.columns:
    df['Remark From Client'] = df['Remark From Client'].replace(
        {'Provision of Certificates Rejected by Candidate': 'Certificated Not Provided by Candidate'}
    )
else:
    print("⚠️ Warning: Column 'Remark From Client' not found in the sheet.")

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd

# Load the cleaned tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')

print("\n==========================================================================================")
print(f" ALL COLUMN NAMES IN YOUR SHEET (Total: {len(df.columns)})")
print("==========================================================================================")

# Loop through and display each column name clearly
for idx, col in enumerate(df.columns, 1):
    print(f"{idx}. {col}")

print("==========================================================================================")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Renaming and standardizing column headers...")

# 3. Define the precise mapping dictionary for corrections
rename_map = {
    'S.no': 'S.No.',
    'Apply Date': 'Date',
    'Phone no.': 'Phone Number',
    'Refered By': 'Referred By',
    'Relevant Experience': 'Relevant Experience (Yrs)',
    'Total Experience': 'Total Experience (Yrs)',
    'Update From Client': 'Client Update',
    'Remark From Client': 'Client Remarks'
}

# Apply the structural rename
df = df.rename(columns=rename_map)

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Columns standardized and saved over 'Daily_Update_Cleaned.xlsx'.")

# --- CONSOLE VALIDATION PREVIEW ---
print("\n==========================================================================================")
print("                           POST-UPDATE COLUMN VERIFICATION                                ")
print("==========================================================================================")
for idx, col in enumerate(df.columns, 1):
    print(f"{idx}. {col}")
print("==========================================================================================")

In [ ]:
import pandas as pd
import re

# 1. Load the spreadsheet (using the updated column names format)
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')

print("\n==========================================================================================")
print("                       🕵️‍♂️ DATA ANOMALY REPORT                            ")
print("==========================================================================================")

# Track if any anomalies are found
anomalies_found = False

# Iterate through each column to test string rules
for col in df.columns:
    # We only check columns with textual entries
    if df[col].dtype == 'object':

        # 1. Check for Unnecessary Whitespaces (Leading, Trailing, or Multiple spaces inside)
        space_condition = df[col].astype(str).str.contains(r'^\s+|\s+$|\s{2,}', regex=True) & df[col].notna()
        space_anomalies = df[space_condition]

        if not space_anomalies.empty:
            anomalies_found = True
            print(f"\n⚠️ Column '{col}' has {len(space_anomalies)} rows with unnecessary/hidden whitespaces:")
            for idx, row in space_anomalies.head(3).iterrows():
                print(f"   • Row Index {idx} (S.No. {row.get('S.No.', idx)}): [Name: {row.get('Name', 'Unknown')}] -> Value: '{row[col]}'")
            if len(space_anomalies) > 3:
                print(f"   ... and {len(space_anomalies) - 3} more rows.")

        # 2. Check for inconsistent Text Styles (Mixed Casing vs common Title case flags)
        # Specifically targeting lower/upper mixtures in standard descriptive columns like 'Location' or 'Referred By'
        if col in ['Location', 'Referred By', 'Skill']:
            casing_condition = df[col].astype(str).apply(lambda x: x.islower() or x.isupper()) & df[col].notna() & ~df[col].astype(str).isin(['N/A', 'NAN', ''])
            casing_anomalies = df[casing_condition]

            if not casing_anomalies.empty:
                anomalies_found = True
                print(f"\n Column '{col}' has {len(casing_anomalies)} rows with non-standard capitalization/text style:")
                for idx, row in casing_anomalies.head(3).iterrows():
                    print(f"   • Row Index {idx}: Value is '{row[col]}'")

        # 3. Check for Hidden Typos or trailing characters (like 'by by' typo variations)
        typo_condition = df[col].astype(str).str.contains(r'\bby by\b', flags=re.IGNORECASE, regex=True)
        typo_anomalies = df[typo_condition]

        if not typo_anomalies.empty:
            anomalies_found = True
            print(f"\n Column '{col}' contains potential copy-paste text duplication ('by by'):")
            for idx, row in typo_anomalies.iterrows():
                print(f"   • Row Index {idx}: Value is '{row[col]}'")

if not anomalies_found:
    print("\n✅ Clean Audit! No messy whitespaces, broken spacing, or weird casing structures detected.")
print("==========================================================================================")

In [ ]:
import pandas as pd
import numpy as np
import re

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Executing whitespace corrections on Name, Phone Number, and Referred By...")

# 3. Apply Targeted Cleansing Rules (leaving 'Skill' untouched)

# Fix Name: Collapse internal multiple spaces and remove any leading/trailing spaces
if 'Name' in df.columns:
    df['Name'] = df['Name'].astype(str).apply(lambda x: re.sub(r'\s+', ' ', x).strip() if pd.notna(x) else x)

# Fix Phone Number: Strip standard and special invisible whitespaces (\xa0)
if 'Phone Number' in df.columns:
    df['Phone Number'] = df['Phone Number'].astype(str).apply(lambda x: re.sub(r'^\s+|\s+$', '', x.replace('\xa0', '')) if pd.notna(x) else x)

# Fix Referred By: Strip trailing whitespace
if 'Referred By' in df.columns:
    df['Referred By'] = df['Referred By'].astype(str).apply(lambda x: x.strip() if pd.notna(x) else x)

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Identified anomalies have been repaired and saved to 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Syncing 'Client Update' and 'Client Remarks' columns with recruiter follow-up rules...")

# 3. Apply conditional logical overwrites based on Assigned Recruiter Follow-up
target_cols = ['Client Update', 'Client Remarks']

if 'Assigned Recruiter Follow-up' in df.columns:
    for col in target_cols:
        if col in df.columns:
            # Rule A: Map 'N/A (Internal Reject)'
            df.loc[df['Assigned Recruiter Follow-up'] == 'N/A (Internal Reject)', col] = 'Not Selected (Internal Reject)'

            # Rule B: Map 'N/A (Duplicate)'
            df.loc[df['Assigned Recruiter Follow-up'] == 'N/A (Duplicate)', col] = 'Not Selected (Duplicate)'
        else:
            print(f"⚠️ Warning: Column '{col}' missing from sheet.")
else:
    print("❌ Error: 'Assigned Recruiter Follow-up' column missing from sheet.")

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Spreadsheet updated and saved over 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import Alignment

# 1. Load your master tracking sheet using pandas
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Writing data changes to 'Daily_Update_Cleaned.xlsx'...")
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')

print(" Applying alignment styles (Headers: Center, Data Entries: Left)...")

# 3. Load the workbook with openpyxl to apply structural cell alignments
wb = openpyxl.load_workbook("Daily_Update_Cleaned.xlsx")
ws = wb['Sheet3']

# Define the alignments
center_alignment = Alignment(horizontal='center', vertical='center')
left_alignment = Alignment(horizontal='left', vertical='center')

# Iterate through rows and columns to apply alignment rules
for row_idx, row in enumerate(ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column), start=1):
    for cell in row:
        if row_idx == 1:
            # Row 1 corresponds to your Column Headers
            cell.alignment = center_alignment
        else:
            # All other rows correspond to your Data Entries
            cell.alignment = left_alignment

# 4. Save visual styling configurations back directly to the file
wb.save("Daily_Update_Cleaned.xlsx")
print("✅ Success! Layout alignment configurations applied and spreadsheet saved successfully.")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print(" Standardizing SAP BTP / SAP ABAP BTP skill sets...")

# 3. Apply the conditional replacements to the Skill column
if 'Skill' in df.columns:
    # Handle string stripping to prevent hidden space mismatches
    df['Skill'] = df['Skill'].astype(str).str.strip()

    # Map 'SAP ABAP BTP' and 'SAP BTP' directly to 'SAP BTP'
    df['Skill'] = df['Skill'].replace({
        'SAP ABAP BTP': 'SAP BTP',
        'SAP BTP': 'SAP BTP'
    })
else:
    print("❌ Error: 'Skill' column not found in the sheet.")

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Skills standardized and saved over 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd

# 1. Load the cleaned tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

print("\n==========================================================================================")
print(" UNIQUE ENTRIES IN 'REFERRED BY' COLUMN")
print("==========================================================================================")

if 'Referred By' in df.columns:
    # Get the unique values and their total distribution count
    referred_counts = df['Referred By'].value_counts(dropna=False)

    # Clean up the print alignment layout for the console
    for name, count in referred_counts.items():
        display_name = "Blank / Empty Cell" if pd.isna(name) or str(name).strip() == "" else name
        print(f" • {display_name:<30} -> Count: {count} times")
else:
    print("❌ Error: 'Referred By' column not found in the sheet.")

print("==========================================================================================")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load your master tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print("Correcting typo in the 'Referred By' column...")

# 3. Apply the targeted name replacement rule
if 'Referred By' in df.columns:
    df['Referred By'] = df['Referred By'].replace({
        'Abhilasha & Masharrat': 'Abhilasha & Masarrat'
    })
else:
    print("Error: 'Referred By' column not found in the sheet.")

# 4. Save modifications back directly to the file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')
print("✅ Success! Typo corrected and changes saved over 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import Alignment

# 1. Load your master tracking sheet using pandas
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print("Replacing text placeholdlers with empty blank cells in 'Location' column...")

# 3. Target the Location column and replace specified text with NaN (blank)
if 'Location' in df.columns:
    df['Location'] = df['Location'].replace({'Not Provided by Candidate': None})
else:
    print("Error: 'Location' column not found in the sheet.")

# Save data matrix back to sheet
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')

print("Applying openpyxl visual layouts (Headers: Center, Data Entries: Left)...")

# 4. Enforce openpyxl styling engine to preserve text alignment layouts
wb = openpyxl.load_workbook("Daily_Update_Cleaned.xlsx")
ws = wb['Sheet3']

center_alignment = Alignment(horizontal='center', vertical='center')
left_alignment = Alignment(horizontal='left', vertical='center')

for row_idx, row in enumerate(ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column), start=1):
    for cell in row:
        if row_idx == 1:
            cell.alignment = center_alignment
        else:
            cell.alignment = left_alignment

wb.save("Daily_Update_Cleaned.xlsx")
print("✅Success! Location placeholder values cleared and visual alignment structure re-saved.")

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import Alignment

# 1. Load your master tracking sheet using pandas
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print("Clearing text placeholders from experience columns...")

# 3. Target metrics columns and replace text with structural blanks (None)
target_metrics = ['Relevant Experience (Yrs)', 'Total Experience (Yrs)']

for col in target_metrics:
    if col in df.columns:
        df[col] = df[col].replace({'Not Provided by Candidate': None})
    else:
        print(f"Error: Column '{col}' not found in the sheet.")

# Save data matrix back to sheet
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')

print("Applying openpyxl visual layouts (Headers: Center, Data Entries: Left)...")

# 4. Enforce openpyxl styling engine to preserve text alignment layouts
wb = openpyxl.load_workbook("Daily_Update_Cleaned.xlsx")
ws = wb['Sheet3']

center_alignment = Alignment(horizontal='center', vertical='center')
left_alignment = Alignment(horizontal='left', vertical='center')

for row_idx, row in enumerate(ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column), start=1):
    for cell in row:
        if row_idx == 1:
            cell.alignment = center_alignment
        else:
            cell.alignment = left_alignment

wb.save("Daily_Update_Cleaned.xlsx")
print("✅Success! Metrics columns cleared and structural alignment re-saved.")

In [ ]:
import pandas as pd

# 1. Load the cleaned tracking sheet
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

print("\n==========================================================================================")
print("UNIQUE ENTRIES IN 'NOTICE PERIOD' COLUMN")
print("==========================================================================================")

if 'Notice Period' in df.columns:
    # Get the unique values and their total distribution count
    notice_counts = df['Notice Period'].value_counts(dropna=False)

    # Print the breakdown cleanly
    for period, count in notice_counts.items():
        display_period = "Blank / Empty Cell" if pd.isna(period) or str(period).strip() == "" else period
        print(f" • {str(display_period):<35} -> Count: {count} times")
else:
    print("Error: 'Notice Period' column not found in the sheet.")

print("==========================================================================================")

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import Alignment

# 1. Load your master tracking sheet using pandas
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. EMAIL PROTECTION LAYER: Pull raw email values from the source to keep baseline data intact
df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# Create a master email lookup map based on lowercase names
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# Restore the email hyperlink formula for each candidate before modifying data
for idx, row in df.iterrows():
    cand_name_key = str(row.get('Name', '')).strip().lower()
    raw_email = email_lookup.get(cand_name_key)

    if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
        clean_email = str(raw_email).strip().lower()
        df.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
    else:
        df.at[idx, 'Email'] = 'Not Provided by Candidate'

print("Renaming column and standardizing Notice Period timelines...")

# 3. Rename column header and map standard values
if 'Notice Period' in df.columns:
    # Handle whitespace cleaning on values before mapping
    df['Notice Period'] = df['Notice Period'].astype(str).str.strip()

    # Define exact standardization replacement mapping
    notice_map = {
        'Immediate': 0,
        '30 Days': 30,
        '60 Days': 60,
        '90 Days': 90,
        '10 Days': 10,
        '16 Days': 16,
        '20 Days': 20,
        '23 Days': 23,
        '40 Days': 40,
        'Not Provided by Candidate': None,
        'nan': None,
        'None': None
    }

    df['Notice Period'] = df['Notice Period'].map(notice_map)
    df = df.rename(columns={'Notice Period': 'Notice Period (Days)'})
else:
    print("Error: 'Notice Period' column not found in the sheet.")

# Save modified data to excel
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')

print("Applying openpyxl visual layouts (Headers: Center, Data Entries: Left)...")

# 4. Enforce openpyxl styling engine to preserve text alignment layouts
wb = openpyxl.load_workbook("Daily_Update_Cleaned.xlsx")
ws = wb['Sheet3']

center_alignment = Alignment(horizontal='center', vertical='center')
left_alignment = Alignment(horizontal='left', vertical='center')

for row_idx, row in enumerate(ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column), start=1):
    for cell in row:
        if row_idx == 1:
            cell.alignment = center_alignment
        else:
            cell.alignment = left_alignment

wb.save("Daily_Update_Cleaned.xlsx")
print("✅Success! Notice Period values converted to clean numbers and saved.")

In [ ]:
# Check unique values in CTC columns
if 'Current CTC' in df.columns:
    print("\n UNIQUE VALUES IN 'CURRENT CTC':")
    print(df['Current CTC'].value_counts(dropna=False).head(20))

if 'Expected CTC' in df.columns:
    print("\n UNIQUE VALUES IN 'EXPECTED CTC':")
    print(df['Expected CTC'].value_counts(dropna=False).head(20))

In [ ]:
# Replace text placeholder with actual blank cells in memory
if 'Current CTC' in df.columns:
    df['Current CTC'] = df['Current CTC'].replace({'Not Provided by Candidate': None})

if 'Expected CTC' in df.columns:
    df['Expected CTC'] = df['Expected CTC'].replace({'Not Provided by Candidate': None})

In [ ]:
import pandas as pd

# 1. Load the dataset into memory
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. Replace text placeholder with actual blank cells in memory
if 'Current CTC' in df.columns:
    df['Current CTC'] = df['Current CTC'].replace({'Not Provided by Candidate': None})

if 'Expected CTC' in df.columns:
    df['Expected CTC'] = df['Expected CTC'].replace({'Not Provided by Candidate': None})

print("The text placeholder 'Not Provided by Candidate' has been successfully replaced with a blank cell in both the Current CTC and Expected CTC columns.")

In [ ]:
import pandas as pd

# 1. Load the dataset into memory
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. Force text placeholder to completely empty/blank cell characters
if 'Current CTC' in df.columns:
    df['Current CTC'] = df['Current CTC'].replace({'Not Provided by Candidate': ""})

if 'Expected CTC' in df.columns:
    df['Expected CTC'] = df['Expected CTC'].replace({'Not Provided by Candidate': ""})

print(f"Replacement complete. Remaining 'Not Provided by Candidate' text entries in CTC columns: {(df['Current CTC'] == 'Not Provided by Candidate').sum()}")

In [ ]:
import pandas as pd

# 1. Load the dataset
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. Clean hidden spaces and remove the text placeholder
for col in ['Current CTC', 'Expected CTC']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({'Not Provided by Candidate': ""})

# 3. Save the changes back to the Excel file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')

print("The text placeholder 'Not Provided by Candidate' has been successfully replaced with a blank cell and saved over 'Daily_Update_Cleaned.xlsx'.")

In [ ]:
import pandas as pd
import openpyxl
from openpyxl.styles import Alignment

# 1. Load dataframes
df_cleaned = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df_cleaned.columns = [col.strip() for col in df_cleaned.columns]

df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# 2. Build map of original candidate names to raw email strings
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# 3. Re-inject active hyperlink formulas into the cleaning matrix
if 'Email' in df_cleaned.columns:
    for idx, row in df_cleaned.iterrows():
        cand_name_key = str(row.get('Name', '')).strip().lower()
        raw_email = email_lookup.get(cand_name_key)

        if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
            clean_email = str(raw_email).strip().lower()
            df_cleaned.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
        else:
            df_cleaned.at[idx, 'Email'] = 'Not Provided by Candidate'

# Save combined modifications
df_cleaned.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')

# 4. Enforce openpyxl visual formatting rules
wb = openpyxl.load_workbook("Daily_Update_Cleaned.xlsx")
ws = wb['Sheet3']

center_alignment = Alignment(horizontal='center', vertical='center')
left_alignment = Alignment(horizontal='left', vertical='center')

for row_idx, row in enumerate(ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column), start=1):
    for cell in row:
        if row_idx == 1:
            cell.alignment = center_alignment
        else:
            cell.alignment = left_alignment

wb.save("Daily_Update_Cleaned.xlsx")

print("The email hyperlink formulas have been successfully restored, cell alignments have been corrected, and the workbook has been updated.")

In [ ]:
import pandas as pd

# 1. Load the dataset
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. Clean hidden spaces and replace the specified text entries with a blank cell
if 'Client Share Date' in df.columns:
    df['Client Share Date'] = df['Client Share Date'].astype(str).str.strip()
    df['Client Share Date'] = df['Client Share Date'].replace({
        'N/A (Internal Reject)': "",
        'N/A (Duplicate)': "",
        'Not Noted by HR': ""
    })

# 3. Save the changes back to the Excel file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')

print("The text entries 'N/A (Internal Reject)', 'N/A (Duplicate)', and 'Not Noted by HR' have been successfully replaced with a blank cell in the Client Share Date column and saved.")

In [ ]:
import pandas as pd
import openpyxl
from openpyxl.styles import Alignment

# 1. Load dataframes
df_cleaned = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df_cleaned.columns = [col.strip() for col in df_cleaned.columns]

df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# 2. Build map of original candidate names to raw email strings
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# 3. Re-inject active hyperlink formulas into the cleaning matrix
if 'Email' in df_cleaned.columns:
    for idx, row in df_cleaned.iterrows():
        cand_name_key = str(row.get('Name', '')).strip().lower()
        raw_email = email_lookup.get(cand_name_key)

        if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
            clean_email = str(raw_email).strip().lower()
            df_cleaned.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
        else:
            df_cleaned.at[idx, 'Email'] = 'Not Provided by Candidate'

# Save combined modifications
df_cleaned.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')

# 4. Enforce openpyxl visual formatting rules
wb = openpyxl.load_workbook("Daily_Update_Cleaned.xlsx")
ws = wb['Sheet3']

center_alignment = Alignment(horizontal='center', vertical='center')
left_alignment = Alignment(horizontal='left', vertical='center')

for row_idx, row in enumerate(ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column), start=1):
    for cell in row:
        if row_idx == 1:
            cell.alignment = center_alignment
        else:
            cell.alignment = left_alignment

wb.save("Daily_Update_Cleaned.xlsx")

print("The email hyperlink formulas have been successfully restored, cell alignments have been corrected, and the workbook has been updated.")

In [ ]:
import pandas as pd

# 1. Load the dataset
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

# 2. Define targets for both Pandas index values and Excel raw row numbers to be absolutely safe
target_rows = [20, 21, 23, 28, 29]

if 'Client Share Date' in df.columns:
    for idx in target_rows:
        # Check both the direct index and the adjacent index positions to guarantee a hit
        for check_idx in [idx, idx-1, idx-2]:
            if check_idx in df.index:
                val_str = str(df.at[check_idx, 'Client Share Date']).strip()

                # Catch dd/mm/2025 or mm/dd/2025 patterns
                if "2025" in val_str:
                    df.at[check_idx, 'Client Share Date'] = val_str.replace('2025', '2026')

# 3. Save the changes back to the Excel file
df.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')

print("The year has been successfully updated to 2026 for the targeted rows in the Client Share Date column, and the file has been saved.")

In [ ]:
import pandas as pd
import openpyxl
from openpyxl.styles import Alignment

# 1. Load dataframes
df_cleaned = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df_cleaned.columns = [col.strip() for col in df_cleaned.columns]

df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# 2. Build map of original candidate names to raw email strings
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

# 3. Re-inject active hyperlink formulas into the cleaning matrix
if 'Email' in df_cleaned.columns:
    for idx, row in df_cleaned.iterrows():
        cand_name_key = str(row.get('Name', '')).strip().lower()
        raw_email = email_lookup.get(cand_name_key)

        if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
            clean_email = str(raw_email).strip().lower()
            df_cleaned.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
        else:
            df_cleaned.at[idx, 'Email'] = 'Not Provided by Candidate'

# Save combined modifications
df_cleaned.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')

# 4. Enforce openpyxl visual formatting rules
wb = openpyxl.load_workbook("Daily_Update_Cleaned.xlsx")
ws = wb['Sheet3']

center_alignment = Alignment(horizontal='center', vertical='center')
left_alignment = Alignment(horizontal='left', vertical='center')

for row_idx, row in enumerate(ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column), start=1):
    for cell in row:
        if row_idx == 1:
            cell.alignment = center_alignment
        else:
            cell.alignment = left_alignment

wb.save("Daily_Update_Cleaned.xlsx")

print("The email hyperlink formulas have been successfully restored, cell alignments have been corrected, and the workbook has been updated.")

In [ ]:
import pandas as pd

# 1. Load the dataset
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

print("COLUMN DATA TYPE DIAGNOSTICS:")
print("==================================================")

for col in ['Phone Number', 'Email']:
    if col in df.columns:
        print(f"\n Column: '{col}'")
        print(f"   Pandas Data Type: {df[col].dtype}")

        # Breakdown of the actual underlying Python types in that column
        type_counts = df[col].apply(lambda x: type(x).__name__).value_counts()
        print("   Underlying Python types present:")
        for py_type, count in type_counts.items():
            print(f"     - {py_type}: {count} entries")

        print("   Sample values:")
        print(df[col].head(5).to_string(index=True))
    else:
        print(f"\n Column '{col}' not found in the sheet.")

print("==================================================")

In [ ]:
import pandas as pd

# 1. Load the dataset
df = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df.columns = [col.strip() for col in df.columns]

print("COMPLETE SHEET DATA TYPE DIAGNOSTICS:")
print("==================================================")

# Loop through every column in the dataframe
for col in df.columns:
    print(f"\nColumn: '{col}'")
    print(f"   Pandas Data Type: {df[col].dtype}")

    # Breakdown of the actual underlying Python types in that column
    type_counts = df[col].apply(lambda x: type(x).__name__).value_counts()
    print("   Underlying Python types present:")
    for py_type, count in type_counts.items():
        print(f"     - {py_type}: {count} entries")

    print("   Sample values:")
    # Using to_string to prevent truncation of sample data
    print(df[col].head(5).to_string(index=True))
    print("-" * 50)

print("==================================================")

In [ ]:
import pandas as pd
import openpyxl
from openpyxl.styles import Alignment

# 1. Load dataframes
df_cleaned = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df_cleaned.columns = [col.strip() for col in df_cleaned.columns]

df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# ==========================================
# STEP A: FIX THE DATE COLUMNS & TYPOS
# ==========================================
# Convert columns to datetime first to cleanly parse and manipulate the segments
for col in ['Date', 'Client Share Date']:
    if col in df_cleaned.columns:
        df_cleaned[col] = pd.to_datetime(df_cleaned[col], dayfirst=True, errors='coerce')

# Convert Excel Row Numbers to Pandas Index positions (Excel Row - 2 due to headers)
# For Excel Rows 20, 21, 23, 28, 29:
excel_rows = [20, 21, 23, 28, 29]
pandas_indices = [row - 2 for row in excel_rows]

# Fix the Client Share Date typos
if 'Client Share Date' in df_cleaned.columns:
    for idx in pandas_indices:
        if idx in df_cleaned.index and pd.notna(df_cleaned.at[idx, 'Client Share Date']):
            val = df_cleaned.at[idx, 'Client Share Date']
            if val.year == 2025:
                df_cleaned.at[idx, 'Client Share Date'] = val.replace(year=2026)

# Fix the main Date typos specifically for Excel Rows 28 and 29 (Pandas index 26 and 27)
if 'Date' in df_cleaned.columns:
    for idx in [26, 27]: # Corresponds precisely to Excel Rows 28 and 29
        if idx in df_cleaned.index and pd.notna(df_cleaned.at[idx, 'Date']):
            val = df_cleaned.at[idx, 'Date']
            if val.year == 2025:
                df_cleaned.at[idx, 'Date'] = val.replace(year=2026)

# Extract pure date objects (.dt.date converts NaT to None, leaving blank cells in Excel)
for col in ['Date', 'Client Share Date']:
    if col in df_cleaned.columns:
        df_cleaned[col] = df_cleaned[col].dt.date


# ==========================================
# STEP B: HYPERLINK RESTORATION
# ==========================================
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

if 'Email' in df_cleaned.columns:
    for idx, row in df_cleaned.iterrows():
        cand_name_key = str(row.get('Name', '')).strip().lower()
        raw_email = email_lookup.get(cand_name_key)

        if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
            clean_email = str(raw_email).strip().lower()
            df_cleaned.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
        else:
            df_cleaned.at[idx, 'Email'] = ""

# Save merged dataset structure back to file
df_cleaned.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')


# ==========================================
# STEP C: OPENPYXL LAYOUT ALIGNMENTS & FINAL SAVE
# ==========================================
wb = openpyxl.load_workbook("Daily_Update_Cleaned.xlsx")
ws = wb['Sheet3']

center_alignment = Alignment(horizontal='center', vertical='center')
left_alignment = Alignment(horizontal='left', vertical='center')

for row_idx, row in enumerate(ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column), start=1):
    for cell in row:
        if row_idx == 1:
            cell.alignment = center_alignment
        else:
            cell.alignment = left_alignment

# Absolute force write to disk
wb.save("Daily_Update_Cleaned.xlsx")
wb.close()

print("Done")

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.styles import Alignment

# 1. Load dataframes
df_cleaned = pd.read_excel("Daily_Update_Cleaned.xlsx", sheet_name='Sheet3')
df_cleaned.columns = [col.strip() for col in df_cleaned.columns]

df_orig = pd.read_excel("Daily Update.xlsx", sheet_name='Sheet3')
df_orig.columns = [col.strip() for col in df_orig.columns]

# ==========================================
# STEP A: SAFE DATE CONVERSION & TYPO FIXES
# ==========================================
for col in ['Date', 'Client Share Date']:
    if col in df_cleaned.columns:
        # Check if the column is already numeric/serial numbers
        if pd.api.types.is_numeric_dtype(df_cleaned[col]):
            df_cleaned[col] = pd.to_datetime(df_cleaned[col], origin='1899-12-30', unit='D', errors='coerce')
        else:
            # If it's already text strings or standard dates, parse normally
            df_cleaned[col] = pd.to_datetime(df_cleaned[col], dayfirst=True, errors='coerce')

# Convert Excel Row Numbers to Pandas Index positions (Excel Row - 2 due to headers)
excel_rows = [20, 21, 23, 28, 29]
pandas_indices = [row - 2 for row in excel_rows]

# Direct year overrides to 2026 for the requested target rows
if 'Client Share Date' in df_cleaned.columns:
    for idx in pandas_indices:
        if idx in df_cleaned.index and pd.notna(df_cleaned.at[idx, 'Client Share Date']):
            val = df_cleaned.at[idx, 'Client Share Date']
            if val.year == 2025:
                df_cleaned.at[idx, 'Client Share Date'] = val.replace(year=2026)

if 'Date' in df_cleaned.columns:
    for idx in [26, 27]: # Precisely targets Excel Rows 28 and 29
        if idx in df_cleaned.index and pd.notna(df_cleaned.at[idx, 'Date']):
            val = df_cleaned.at[idx, 'Date']
            if val.year == 2025:
                df_cleaned.at[idx, 'Date'] = val.replace(year=2026)

# Extract pure date objects (.dt.date converts NaT to None, keeping Excel rows completely empty)
for col in ['Date', 'Client Share Date']:
    if col in df_cleaned.columns:
        df_cleaned[col] = df_cleaned[col].dt.date


# ==========================================
# STEP B: BLANK OUT "Not Noted by HR" CORES
# ==========================================
df_cleaned.replace('Not Noted by HR', "", inplace=True)


# ==========================================
# STEP C: HYPERLINK RESTORATION
# ==========================================
email_lookup = dict(zip(
    df_orig['Name'].astype(str).str.strip().str.lower(),
    df_orig['Email']
))

if 'Email' in df_cleaned.columns:
    for idx, row in df_cleaned.iterrows():
        cand_name_key = str(row.get('Name', '')).strip().lower()
        raw_email = email_lookup.get(cand_name_key)

        if pd.notna(raw_email) and str(raw_email).strip().lower() not in ['nan', 'none', '']:
            clean_email = str(raw_email).strip().lower()
            df_cleaned.at[idx, 'Email'] = f'=HYPERLINK("mailto:{clean_email}", "{clean_email}")'
        else:
            df_cleaned.at[idx, 'Email'] = ""

# Save structural alterations back to Excel
df_cleaned.to_excel("Daily_Update_Cleaned.xlsx", index=False, sheet_name='Sheet3')


# ==========================================
# STEP D: OPENPYXL FORMATTING & LAYOUT RULES
# ==========================================
wb = openpyxl.load_workbook("Daily_Update_Cleaned.xlsx")
ws = wb['Sheet3']

center_alignment = Alignment(horizontal='center', vertical='center')
left_alignment = Alignment(horizontal='left', vertical='center')

for row_idx, row in enumerate(ws.iter_rows(min_row=1, max_row=ws.max_row, min_col=1, max_col=ws.max_column), start=1):
    for cell in row:
        if row_idx == 1:
            cell.alignment = center_alignment
        else:
            cell.alignment = left_alignment

wb.save("Daily_Update_Cleaned.xlsx")
wb.close()

print("Done")

In [ ]:
import openpyxl
import re

# 1. Load the workbook and select Sheet3
wb = openpyxl.load_workbook('Daily_Update_Cleaned.xlsx')
ws = wb['Sheet3']

# 2. Locate the 'Email' column index
headers = [cell.value for cell in ws[1]]
if 'Email' in headers:
    email_col_idx = headers.index('Email') + 1

    # 3. Clean every cell in the Email column
    for row in range(2, ws.max_row + 1):
        cell = ws.cell(row=row, column=email_col_idx)

        # Type 1: Remove underlying clickable hyperlink property
        if cell.hyperlink:
            cell.hyperlink = None

        # Type 2: Extract text from Excel formulas like =HYPERLINK("mailto:...", "...")
        if isinstance(cell.value, str) and cell.value.upper().startswith('=HYPERLINK'):
            # Find everything inside quotation marks
            quoted_strings = re.findall(r'"([^"]*)"', cell.value)
            if quoted_strings:
                # Take the display text (usually the last quoted item) and clean it
                clean_email = quoted_strings[-1].replace('mailto:', '').strip()
                cell.value = clean_email

        # Reset the styling to standard black text without underlines
        cell.font = openpyxl.styles.Font(color="000000", underline="none")

    # 4. Save the corrected file
    wb.save('Daily_Update_Cleaned.xlsx')
    print("Successfully converted all emails to pure plain text.")
else:
    print("Could not find a column named 'Email' in Sheet3.")